# Day 10 - Seq2Seq & Attention Forecasting

## RetailMind AI Increment

Day 10 upgrades Day 9's next-step LSTM into a **multi-horizon encoder-decoder forecaster**. A 56-day history window is encoded once, a decoder emits the next 14 daily demand values, and temporal attention assigns a relevance weight to every historical day for each forecast step.

### Business objective
Procurement teams need more than tomorrow's demand. A 14-day path supports replenishment lead times, promotion preparation, and stock allocation while attention provides an explanation of which historical periods influenced each horizon.

## Architecture and leakage controls

- **Input:** 56 daily sales values for one Rossmann store.
- **Output:** 14 future daily sales values.
- **Encoder:** LSTM with 64 units and full hidden-state sequence.
- **Decoder:** LSTM with 64 units producing one state per future day.
- **Attention:** additive temporal attention over the encoder sequence.
- **Split:** latest 56 days for test and preceding 56 days for validation.
- **Normalization:** mean and standard deviation fitted only on the training period.
- **Comparison:** persistence baseline versus Seq2Seq + attention using MAE, RMSE, and MAPE.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day10_seq2seq_attention_forecasting as day10

print("TensorFlow version:", day10.tf.__version__)
print("Look-back:", day10.LOOKBACK, "days")
print("Forecast horizon:", day10.HORIZON, "days")

## 1. Load Rossmann sales

Place `train.csv` in the repository root. The loader uses Python's standard `csv` module so the Day 10 implementation remains within the TensorFlow/Keras library constraint. The first store is selected by default and missing calendar days are filled with zero sales.

In [ ]:
dates, series, store_id = day10.load_daily_sales(repo_root / "train.csv")
print("Store:", store_id)
print("Date range:", dates[0], "to", dates[-1])
print("Daily observations:", len(series))
print("First 10 sales values:", series[:10].numpy())

## 2. Create chronological Seq2Seq windows

Each supervised example has shape `(56, 1)` and maps to a 14-value target. The split is based on target dates, preventing future targets from entering training. Normalization statistics are fitted only on the training period.

In [ ]:
data = day10.prepare_data(series)
X_train, y_train, X_val, y_val, X_test, y_test, mean, std = data
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)
print("Training mean/std:", float(mean), float(std))

## 3. Build the encoder-decoder with attention

The encoder returns all 56 hidden states plus its final state. The decoder begins with the encoder's final state and produces 14 decoder states. For every forecast day, additive attention scores all encoder days, applies softmax, and creates a weighted history context. The decoder state and context are concatenated before the final forecast layer.

In [ ]:
model, explainer = day10.build_seq2seq_attention()
model.summary()

## 4. Train with early stopping

Early stopping restores the best validation weights and `ReduceLROnPlateau` lowers the learning rate when validation loss stalls. CPU training is intentionally small.

In [ ]:
callbacks = [
    day10.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    day10.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", patience=2, factor=0.5
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=day10.EPOCHS,
    batch_size=day10.BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)
print("Epochs completed:", len(history.history["loss"]))

## 5. Compare multi-step accuracy

The persistence baseline repeats the most recent observed demand across all 14 future days. This approximates carrying Day 9's single-step value forward and provides an honest multi-horizon comparison.

In [ ]:
pred_norm, attention_weights = explainer.predict(X_test, verbose=0)
actual = y_test * std + mean
predicted = pred_norm * std + mean
baseline = day10.tf.repeat(
    (X_test[:, -1, 0] * std + mean)[:, None], day10.HORIZON, axis=1
)

baseline_scores = day10.metrics(actual, baseline)
model_scores = day10.metrics(actual, predicted)

print("Model                    MAE       RMSE      MAPE")
print("Persistence baseline  %8.2f  %8.2f  %7.2f%%" % baseline_scores)
print("Seq2Seq + attention   %8.2f  %8.2f  %7.2f%%" % model_scores)
print("MAE improvement:", round(baseline_scores[0] - model_scores[0], 2))

## 6. Inspect the first 14-day forecast

In [ ]:
print("Day | Actual | Predicted | Absolute Error")
for step in range(day10.HORIZON):
    a = float(actual[0, step])
    p = float(predicted[0, step])
    print(f"{step+1:3d} | {a:6.1f} | {p:9.1f} | {abs(a-p):14.1f}")

## 7. Explain and visualize attention

The attention matrix has one row per forecast day and one column per historical day. Each row sums to approximately 1. The highest-weight days identify which parts of the 56-day history the decoder emphasized. The TensorFlow-only helper writes a grayscale PNG heatmap, avoiding an additional plotting dependency.

In [ ]:
first_attention = attention_weights[0]
print("Attention shape:", first_attention.shape)
print("Row sums:", day10.tf.reduce_sum(first_attention, axis=1).numpy().round(4))

for step in range(day10.HORIZON):
    top = day10.tf.argsort(first_attention[step], direction="DESCENDING")[:3].numpy()
    relative_days = [int(i) - day10.LOOKBACK + 1 for i in top]
    print(f"Forecast day {step+1:2d}: strongest history offsets {relative_days}")

day10.save_attention_heatmap(first_attention, repo_root / day10.ATTENTION_IMAGE_PATH)
print("Saved:", repo_root / day10.ATTENTION_IMAGE_PATH)

## 8. Procurement integration and conclusions

The Day 10 model upgrades the demand module from one-step output to a 14-day procurement horizon. The saved `.keras` model can produce a complete replenishment path from one history window. The forecast CSV supports downstream reorder calculations, while the attention CSV and heatmap provide evidence about which past days influenced each predicted day.

A deployment decision should require the Seq2Seq model to beat the persistence baseline on untouched test MAE and RMSE. Attention should not be treated as proof of causality; it is a model-internal relevance signal. Additional improvements could include known promotion calendars, holidays, store-open flags, item-level identifiers, probabilistic quantile forecasts, and rolling-origin evaluation across several time periods.

In [ ]:
# Run the complete production-style pipeline and save all artifacts.
# Uncomment after train.csv is in the repository root.
# day10.main()